# Getting Started with MARA RoboSim

This notebook walks through MARA RoboSim, the PyBullet manipulation
suite shipped with [predicators](https://github.com/Learning-and-Intelligent-Systems/predicators):
discovering available environments, creating one, taking actions,
and rendering a video — all through the standard Gymnasium API.

## Installation

From the predicators repo root:

```bash
pip install -e .
```

This installs the agent solvers and MARA RoboSim together. See
[`predicators/envs/README.md`](../predicators/envs/README.md) for
more details.

In [ ]:
import matplotlib

matplotlib.use("agg")

import numpy as np
import matplotlib.pyplot as plt

from predicators import utils
from predicators.envs import gymnasium_wrapper as mara_robosim

# MARA RoboSim envs read configuration from predicators.settings.CFG.
# `reset_config` applies parser defaults so we can use the envs as a
# library without going through main.py's command-line interface.
utils.reset_config({"num_train_tasks": 1, "num_test_tasks": 1})

## Discovering Available Environments

MARA RoboSim provides 15 PyBullet-based robotic manipulation environments.
Let's register them all and see what's available.

In [ ]:
mara_robosim.register_all_environments()

env_ids = sorted(mara_robosim.get_all_env_ids())
print(f"{len(env_ids)} environments available:\n")
for eid in env_ids:
    print(f"  {eid}")

## Creating an Environment

We'll use `mara/Blocks-v0`, an environment where a Fetch robot
must stack and arrange blocks on a tabletop.

In [ ]:
env = mara_robosim.make("mara/Blocks-v0", render_mode="rgb_array")
obs, info = env.reset()

frame = env.render()
if frame is not None:
    plt.imshow(frame)
    plt.axis("off")
    plt.title("mara/Blocks-v0")
    plt.show()
else:
    print("(rendering not available in this configuration)")

## Exploring the Observation and Action Spaces

MARA RoboSim environments follow the Gymnasium API. Observations and
actions are continuous-valued numpy arrays.

In [ ]:
print("Observation shape:", env.observation_space.shape)
print("Action shape:     ", env.action_space.shape)
print()

action = env.action_space.sample()
obs, reward, terminated, truncated, info = env.step(action)
print("Sample action:", np.round(action, 3))
print("Reward:       ", reward)
print("Terminated:   ", terminated)

## Inspecting the Structured State

Beyond the flat observation vector, the `info` dict exposes the full
object-centric `State` and whether the goal has been reached.

In [ ]:
state = info["state"]
print("Goal reached:", info["goal_reached"])
print()
print(state.pretty_str())

## Rendering a Video

Let's collect frames from random actions and display them as an animated GIF.

In [ ]:
from io import BytesIO

from IPython.display import Image
from PIL import Image as PILImage

obs, info = env.reset()
frames = []
frame = env.render()
if frame is not None:
    frames.append(frame)

for _ in range(50):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    frame = env.render()
    if frame is not None:
        frames.append(frame)
    if terminated or truncated:
        break

if frames:
    pil_frames = [PILImage.fromarray(f) for f in frames]
    buf = BytesIO()
    pil_frames[0].save(
        buf,
        format="GIF",
        save_all=True,
        append_images=pil_frames[1:],
        duration=100,
        loop=0,
    )
    Image(data=buf.getvalue(), format="gif")
else:
    print("(no frames captured — rendering may not be available)")

## Cleanup

In [ ]:
env.close()